# Security Data — Logistic Regression (Multiple Logistic Regression)

## Dataset
Data sourced from `security_data_10000.csv` (synthetic data generated from original class dataset).

| Column | Description |
|--------|-------------|
| Sector | Industry sector of the company |
| CEO_Gender | Gender of the CEO (Male/Female) |
| Size | Company size (Large, Medium, Small) |
| Security_Invest | Security investment amount |
| Security_Breach_Att | Number of security breach attempts |
| Succ_Sec_Breaches | Number of successful security breaches |
| Sec_Rating | Overall security rating (High/Medium/Low) |
| CEO_Sec_Exp | CEO's level of security experience (High/Medium/Low) |
| LOT_in_Business | Length of time in business (years) |
| Stock_Market | Whether the company is listed on the stock market (Yes/No) |

---

## Objective
Predict whether a company's CEO is **Male or Female** using all available features.

**Target variable:** `CEO_Gender` (Male = True / Female = False)

---

## Data Analysis Plan

1. **Load Data** — Read CSV into DataFrame
2. **Train-Test Split** — Split before any feature engineering
3. **Create Binary Label** — Encode `CEO_Gender` as True/False
4. **Drop Target from Features** — Remove `CEO_Gender` from input features
5. **Encode Categorical Features** — One-hot encode remaining categorical columns
6. **Fit Logistic Regression** — Use `class_weight='balanced'` to handle class imbalance
7. **Evaluate** — Confusion matrix and classification report on train and test

In [103]:
#LOAD THE DATASET


import pandas as pd

security_df = pd.read_csv("security_data_10000.csv")

security_df.head()

,Sector,CEO_Gender,Size,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,Sec_Rating,CEO_Sec_Exp,LOT_in_Business,Stock_Market
0,Hospitality,Male,Large,186,80,38,High,High,11,No
1,Hospitality,Female,Large,229,72,42,Medium,Medium,21,Yes
2,Hospitality,Male,Small,108,78,35,High,Medium,15,Yes
3,Hospitality,Male,Large,210,70,35,Medium,Low,14,No
4,Banking,Male,Small,34,19,3,Low,High,3,No


## Step 1 — Load Dataset

In [104]:
from sklearn.model_selection import train_test_split
security_df_train, security_df_test = train_test_split(security_df, test_size=0.2, random_state=42)


## Step 2 — Train-Test Split
Split performed **before** any feature engineering to prevent data leakage.

In [105]:
security_df_train.describe()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
count,8000.000000,8000.000000,8000.000000,8000.000000
mean,80.993500,60.310875,24.949625,13.022375
std,82.579267,73.817377,36.353106,7.190295
min,10.000000,5.000000,0.000000,1.000000
25%,28.000000,17.000000,2.000000,7.000000
50%,50.000000,32.000000,12.000000,13.000000
75%,99.000000,59.000000,26.000000,19.000000
max,450.000000,330.000000,211.000000,25.000000


## Step 3 — Exploratory Data Analysis (EDA)

In [106]:
security_df_train_label = (security_df_train["CEO_Gender"] == 'Male')

security_df_test_label = (security_df_test["CEO_Gender"] =='Male')

## Step 4 — Create Binary Label
Encode `CEO_Gender` as a boolean: `True` = Male, `False` = Female.
This will be the target variable for the logistic regression model.

In [107]:
security_df_train.shape

(8000, 10)

In [108]:
security_df_train = security_df_train.drop("CEO_Gender", axis=1)
security_df_train.shape

(8000, 9)

## Step 5 — Drop Target Column from Features
`CEO_Gender` must be removed from the input features — it is the target variable, not a predictor.

In [109]:
security_df_test = security_df_test.drop("CEO_Gender", axis=1)
security_df_test.shape



(2000, 9)

In [110]:
security_df_train.describe()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
count,8000.000000,8000.000000,8000.000000,8000.000000
mean,80.993500,60.310875,24.949625,13.022375
std,82.579267,73.817377,36.353106,7.190295
min,10.000000,5.000000,0.000000,1.000000
25%,28.000000,17.000000,2.000000,7.000000
50%,50.000000,32.000000,12.000000,13.000000
75%,99.000000,59.000000,26.000000,19.000000
max,450.000000,330.000000,211.000000,25.000000


In [111]:

security_df_train_cat = pd.get_dummies(security_df_train[["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "Stock_Market"]],dtype=int)

security_df_test_cat = pd.get_dummies(security_df_test[["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "Stock_Market"]],dtype=int)

security_df_train_final = pd.concat([security_df_train, security_df_train_cat], axis=1)

security_df_test_final = pd.concat([security_df_test, security_df_test_cat], axis=1)

security_df_train_final = security_df_train_final.drop(["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "Stock_Market"], axis=1)

security_df_test_final = security_df_test_final.drop(["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "Stock_Market"], axis=1)



## Step 6 — Encode Categorical Features
One-hot encode all remaining categorical columns. `dtype=int` ensures 0/1 output instead of True/False.
Original categorical columns are dropped after encoding to avoid duplication.

In [112]:
security_df_train_final.head()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes
9254,123,78,26,16,0,0,1,1,0,0,1,0,0,0,0,1,0,1
1561,419,250,90,15,0,0,1,0,1,0,0,1,0,1,0,0,0,1
1670,28,8,0,15,1,0,0,0,0,1,0,0,1,1,0,0,0,1
6087,86,49,26,18,0,1,0,1,0,0,1,0,0,0,0,1,1,0
6669,125,16,5,10,0,1,0,0,1,0,0,0,1,1,0,0,0,1


## Step 7 — Fit Logistic Regression
`class_weight='balanced'` corrects for class imbalance (more Male than Female CEOs in the dataset).
Without this, the model defaults to predicting the majority class only.

In [113]:
from sklearn.linear_model import LogisticRegression

logistic_model = log_reg = LogisticRegression(class_weight="balanced")

logistic_model.fit(security_df_train_final, security_df_train_label)

security_df_train_label_pred = logistic_model.predict(security_df_train_final)

security_df_test_label_pred = logistic_model.predict(security_df_test_final)



In [114]:
from sklearn.metrics import confusion_matrix, classification_report
print("Confusion Matrix - Train:")

print(confusion_matrix(security_df_train_label, security_df_train_label_pred))

print("Classification Report - Train:")

print(classification_report(security_df_train_label, security_df_train_label_pred))

print("Confusion Matrix - Test:")

print(confusion_matrix(security_df_test_label, security_df_test_label_pred))

print("Classification Report - Test:")

print(classification_report(security_df_test_label, security_df_test_label_pred))

Confusion Matrix - Train:
[[1477 1361]
 [2536 2626]]
Classification Report - Train:
              precision    recall  f1-score   support

       False       0.37      0.52      0.43      2838
        True       0.66      0.51      0.57      5162

    accuracy                           0.51      8000
   macro avg       0.51      0.51      0.50      8000
weighted avg       0.56      0.51      0.52      8000

Confusion Matrix - Test:
[[363 335]
 [645 657]]
Classification Report - Test:
              precision    recall  f1-score   support

       False       0.36      0.52      0.43       698
        True       0.66      0.50      0.57      1302

    accuracy                           0.51      2000
   macro avg       0.51      0.51      0.50      2000
weighted avg       0.56      0.51      0.52      2000



## Step 8 — Evaluate
- **Confusion matrix** — shows True/False Positives and Negatives
- **Precision** — of all predicted Male/Female, how many were correct
- **Recall** — of all actual Male/Female, how many were caught
- **F1-score** — harmonic mean of precision and recall